<a href="https://colab.research.google.com/github/Parag003/Ml_lab/blob/main/LAB_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.model_selection import KFold, train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

url = "/content/USA_Housing.csv"
data = pd.read_csv(url)

X = data.drop(columns=["Price"])
y = data["Price"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kf = KFold(n_splits=5, shuffle=True)

best_r2 = -float('inf')
best_beta = None

for train_index, test_index in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y[train_index], y[test_index]

    model = LinearRegression()
    model.fit(X_train, y_train)

    beta = model.coef_

    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)

    if r2 > best_r2:
        best_r2 = r2
        best_beta = beta

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)
model.coef_ = best_beta
model.intercept_ = y_train.mean()
y_pred = model.predict(X_test)
r2_final = r2_score(y_test, y_pred)
print(f"Best R^2 Score using K-Fold CV: {best_r2}")
print(f"Final R^2 Score using best beta on 70% training data: {r2_final}")


Best R^2 Score using K-Fold CV: 0.9253705612842588
Final R^2 Score using best beta on 70% training data: 0.9145374791222222


#QUESTION 2

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.44, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.682, random_state=42)

def gradient_descent(X, y, learning_rate, iterations):
    m, n = X.shape
    beta = np.zeros(n)
    cost_history = np.zeros(iterations)

    for i in range(iterations):
        prediction = X.dot(beta)
        error = prediction - y
        cost = (1 / (2 * m)) * np.sum(error ** 2)
        cost_history[i] = cost
        gradient = (1 / m) * X.T.dot(error)
        beta -= learning_rate * gradient
    return beta, cost_history

learning_rates = [0.001, 0.01, 0.1, 1]
best_r2_val = -float('inf')
best_beta_val = None

for lr in learning_rates:
    beta, cost_history = gradient_descent(X_train, y_train, lr, 1000)
    y_pred_val = X_val.dot(beta)
    r2_val = r2_score(y_val, y_pred_val)

    if r2_val > best_r2_val:
        best_r2_val = r2_val
        best_beta_val = beta

y_pred_test = X_test.dot(best_beta_val)
r2_test = r2_score(y_test, y_pred_test)
print(f"Best R^2 Score using Gradient Descent (Validation Set): {best_r2_val}")
print(f"Test R^2 Score using best beta: {r2_test}")


Best R^2 Score using Gradient Descent (Validation Set): -11.806114754720713
Test R^2 Score using best beta: -11.798107969777671


#Question 3


In [4]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error

car_data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data"
column_names = ["symboling", "normalized_losses", "make", "fuel_type", "aspiration", "num_doors", "body_style",
                "drive_wheels", "engine_location", "wheel_base", "length", "width", "height", "curb_weight",
                "engine_type", "num_cylinders", "engine_size", "fuel_system", "bore", "stroke", "compression_ratio",
                "horsepower", "peak_rpm", "city_mpg", "highway_mpg", "price"]

car_data = pd.read_csv(car_data_url, names=column_names, na_values="?")

car_data.fillna(car_data.mean(), inplace=True)

car_data.dropna(subset=["price"], inplace=True)

label_cols = ['make', 'aspiration', 'engine_location', 'fuel_type']
le = LabelEncoder()
for col in label_cols:
    car_data[col] = le.fit_transform(car_data[col])

car_data = pd.get_dummies(car_data, columns=["body_style", "drive_wheels"])

X_car = car_data.drop(columns=["price"])
y_car = car_data["price"]

scaler_car = StandardScaler()
X_scaled_car = scaler_car.fit_transform(X_car)

X_train_car, X_test_car, y_train_car, y_test_car = train_test_split(X_scaled_car, y_car, test_size=0.3, random_state=42)
model_car = LinearRegression()
model_car.fit(X_train_car, y_train_car)

y_pred_car = model_car.predict(X_test_car)
mse_car = mean_squared_error(y_test_car, y_pred_car)
print(f"Car Price Prediction MSE: {mse_car}")

pca = PCA(n_components=5)
X_pca = pca.fit_transform(X_scaled_car)

X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(X_pca, y_car, test_size=0.3, random_state=42)
model_pca = LinearRegression()
model_pca.fit(X_train_pca, y_train_pca)

y_pred_pca = model_pca.predict(X_test_pca)
mse_pca = mean_squared_error(y_test_pca, y_pred_pca)
print(f"Car Price Prediction MSE after PCA: {mse_pca}")


TypeError: can only concatenate str (not "int") to str